# 13b_control_experiment.ipynb — XGBoost AFT ชนะ stacking เพราะอะไร?

ใน notebook 13, XGBoost AFT ชนะ stacking (label any) ทุก metric อย่างมีนัยสำคัญ (ROC-AUC +0.013) แต่มี **2 อย่างที่ต่างกันพร้อมกัน**:

| | Stacking (notebook 11) | XGBoost AFT (notebook 13) |
|---|---|---|
| ข้อมูล outcome | binary (2 ระดับ) | 3 ช่วงเวลา (`<30` / `>30` / `NO`) |
| Hyperparameter | lr 0.01, 300 ต้นไม้ **คงที่** (tune มาสำหรับ label 30d) | lr 0.03, ~1,100 ต้นไม้ด้วย **early stopping** |

ยังสรุปไม่ได้ว่า "survival ดีกว่า" จนกว่าจะแยกสองปัจจัยออกจากกัน → เทรน **XGBoost binary ด้วย hyperparameter และ early stopping เดียวกับ AFT ทุกประการ** ต่างกันแค่ objective

| ผลลัพธ์ | แปลว่า |
|---|---|
| binary tuned ≈ AFT | ผลดีขึ้นมาจาก**การ tune** ไม่ใช่ survival → stacking เดิม underfit |
| binary tuned < AFT อย่างมีนัยสำคัญ | ผลดีขึ้นมาจาก**ข้อมูลเวลา 3 ช่วง** ที่ survival ใช้ได้แต่ binary ใช้ไม่ได้ |

In [ ]:
RANDOM_STATE = 42
RAW_PATH = "../data/diabetic_data.csv"
SPLIT_PATH = "../outputs/split_assignment.csv"
SUMMARY_13 = "../outputs/13_survival/summary.json"
OUT_DIR = "../outputs/13b_control"
H = 365
N_BOOT_TEST = 300

import os, json, warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss
from readmission_features import load_prepared, attach_split
os.makedirs(OUT_DIR, exist_ok=True)

X, _, groups, meta, _ = load_prepared(RAW_PATH, "any")
split = attach_split(meta, SPLIT_PATH)
cat = meta["readmitted_raw"].map({"<30": 0, ">30": 1, "NO": 2}).values
idx = {s: np.where(split == s)[0] for s in ["train", "val", "test"]}
Xs = {s: X.iloc[i].reset_index(drop=True) for s, i in idx.items()}
C = {s: cat[i] for s, i in idx.items()}
g_te = groups[idx["test"]]

SIGMA = json.load(open(SUMMARY_13))["aft_sigma"][str(H)]
print(f"ใช้ σ = {SIGMA} (เลือกไว้แล้วใน notebook 13 สำหรับ H = {H})")

ใช้ σ = 1.5 (เลือกไว้แล้วใน notebook 13 สำหรับ H = 365)


## 1. เทรน 4 โมเดล ด้วย hyperparameter ชุดเดียวกันทั้งหมด
- **AFT** — objective survival (ใช้ 3 ช่วงเวลา)
- **Binary tuned (any)** / **Binary tuned (30d)** — objective logistic
- **Ordinal (2 binary)** — ไม่ได้ใช้ในที่นี้ เพราะ AFT ครอบคลุมแนวคิดนี้แล้ว

In [2]:
COMMON = {"tree_method": "hist", "learning_rate": 0.03, "max_depth": 4,
          "subsample": 0.9, "min_child_weight": 5, "seed": RANDOM_STATE}

def train_es(params, dtr, dva):
    bst = xgb.train({**COMMON, **params}, dtr, num_boost_round=3000, evals=[(dva, "val")],
                    early_stopping_rounds=100, verbose_eval=False)
    return bst, bst.best_iteration + 1

def predict(bst, Xd):
    return bst.predict(xgb.DMatrix(Xd), iteration_range=(0, bst.best_iteration + 1))

# --- AFT ---
def aft_dmatrix(Xd, c):
    d = xgb.DMatrix(Xd)
    d.set_float_info("label_lower_bound", np.select([c == 0, c == 1, c == 2], [0.0, 30.0, float(H)]))
    d.set_float_info("label_upper_bound", np.select([c == 0, c == 1, c == 2], [30.0, float(H), np.inf]))
    return d

aft, n_aft = train_es({"objective": "survival:aft", "eval_metric": "aft-nloglik",
                       "aft_loss_distribution": "extreme", "aft_loss_distribution_scale": SIGMA},
                      aft_dmatrix(Xs["train"], C["train"]), aft_dmatrix(Xs["val"], C["val"]))
pt = predict(aft, Xs["test"])
def aft_cdf(t):
    return -np.expm1(-np.exp(np.clip((np.log(t) - np.log(pt)) / SIGMA, -30, 30)))

# --- Binary tuned ---
def binary(label_fn):
    dtr = xgb.DMatrix(Xs["train"], label=label_fn(C["train"]).astype(int))
    dva = xgb.DMatrix(Xs["val"], label=label_fn(C["val"]).astype(int))
    return train_es({"objective": "binary:logistic", "eval_metric": "logloss"}, dtr, dva)

bin_any, n_any = binary(lambda c: c <= 1)
bin_30, n_30 = binary(lambda c: c == 0)
p_any, p_30 = predict(bin_any, Xs["test"]), predict(bin_30, Xs["test"])

print(f"จำนวนต้นไม้ (early stopping): AFT={n_aft}, binary any={n_any}, binary 30d={n_30}")

# score สำหรับ (label any, label 30d, ordinal risk)
scores = {
    "XGB AFT (survival)": (aft_cdf(H), aft_cdf(30), -np.log(pt)),
    "XGB binary tuned (any)": (p_any, p_any, p_any),
    "XGB binary tuned (30d)": (p_30, p_30, p_30),
}

จำนวนต้นไม้ (early stopping): AFT=1155, binary any=1009, binary 30d=879


## 2. เปรียบเทียบบน test + paired bootstrap (ตามผู้ป่วย)
ผลต่างทุกค่าเทียบกับ **XGB binary tuned (any)** ซึ่งเป็น control ที่ยุติธรรมที่สุดของ AFT

In [3]:
c_te = C["test"]
y_any, y_30 = (c_te <= 1).astype(int), (c_te == 0).astype(int)

def ordinal_cindex(c, risk):
    num = den = 0.0
    for a, b in [(0, 1), (0, 2), (1, 2)]:
        m = (c == a) | (c == b)
        na, nb = (c == a).sum(), (c == b).sum()
        num += na * nb * roc_auc_score(c[m] == a, risk[m]); den += na * nb
    return num / den

def metrics(sa, s30, r, ii=slice(None)):
    return {"ROC_AUC_any": roc_auc_score(y_any[ii], sa[ii]),
            "PR_AUC_any": average_precision_score(y_any[ii], sa[ii]),
            "ROC_AUC_30d": roc_auc_score(y_30[ii], s30[ii]),
            "PR_AUC_30d": average_precision_score(y_30[ii], s30[ii]),
            "C_index": ordinal_cindex(c_te[ii], r[ii])}

point = pd.DataFrame({k: metrics(*v) for k, v in scores.items()}).T
print(point.round(4).to_string())
print(f"\nBrier (any): AFT={brier_score_loss(y_any, scores['XGB AFT (survival)'][0]):.4f}  "
      f"binary={brier_score_loss(y_any, p_any):.4f}")

ref = "XGB binary tuned (any)"
rng = np.random.default_rng(RANDOM_STATE)
uniq = np.unique(g_te)
rows = pd.Series(np.arange(len(g_te))).groupby(g_te).apply(np.array)
deltas = {k: [] for k in scores if k != ref}
for b in range(N_BOOT_TEST):
    ii = np.concatenate(rows.loc[rng.choice(uniq, len(uniq), replace=True)].values)
    r = {k: metrics(*v, ii=ii) for k, v in scores.items()}
    for k in deltas:
        deltas[k].append({m: r[k][m] - r[ref][m] for m in r[k]})

out = {}
for k, lst in deltas.items():
    d = pd.DataFrame(lst)
    out[k] = {m: f"{point.loc[k, m] - point.loc[ref, m]:+.4f} ({d[m].quantile(.025):+.4f}, {d[m].quantile(.975):+.4f})"
                 + (" *" if d[m].quantile(.025) > 0 or d[m].quantile(.975) < 0 else "")
              for m in d.columns}
delta_table = pd.DataFrame(out).T
print(f"\nผลต่างเทียบกับ {ref} + 95% CI (* = มีนัยสำคัญ):")
print(delta_table.to_string())

point.round(4).to_csv(f"{OUT_DIR}/point_metrics.csv")
delta_table.to_csv(f"{OUT_DIR}/delta_vs_binary_any.csv")

                        ROC_AUC_any  PR_AUC_any  ROC_AUC_30d  PR_AUC_30d  C_index
XGB AFT (survival)           0.6937      0.6585       0.6548      0.2179   0.6744
XGB binary tuned (any)       0.6946      0.6595       0.6488      0.2111   0.6739
XGB binary tuned (30d)       0.6495      0.6084       0.6795      0.2339   0.6444

Brier (any): AFT=0.2204  binary=0.2201

ผลต่างเทียบกับ XGB binary tuned (any) + 95% CI (* = มีนัยสำคัญ):
                                         ROC_AUC_any                    PR_AUC_any                   ROC_AUC_30d                    PR_AUC_30d                       C_index
XGB AFT (survival)        -0.0009 (-0.0021, +0.0004)    -0.0011 (-0.0029, +0.0012)  +0.0061 (+0.0040, +0.0083) *  +0.0068 (+0.0024, +0.0117) *    +0.0005 (-0.0006, +0.0017)
XGB binary tuned (30d)  -0.0451 (-0.0507, -0.0389) *  -0.0512 (-0.0585, -0.0437) *  +0.0307 (+0.0219, +0.0397) *  +0.0228 (+0.0108, +0.0374) *  -0.0295 (-0.0349, -0.0242) *


## 3. อ่านผล
- ดูแถว **XGB AFT (survival)** คอลัมน์ `ROC_AUC_any`:
  - ถ้า CI คร่อม 0 → ข้อได้เปรียบใน notebook 13 มาจาก **การ tune** → ข้อสรุปคือ "stacking เดิม underfit กับ label ใหม่" และควร re-tune stacking
  - ถ้าบวกและมี * → ข้อมูลเวลา 3 ช่วงช่วยจริง → **survival formulation คือ contribution ของเปเปอร์**
- เทียบ `XGB binary tuned (any)` กับ stacking เดิมใน notebook 13 (ROC-AUC any = 0.6806) เพื่อดูว่าการ tune อย่างเดียวให้ผลเท่าไร